# Notebook 13 — Final Freeze and Revised Manuscript Assets

## Grain AI–MDPI

This notebook consolidates the **FULL**, reviewer-driven reanalysis produced by Notebook 12. It does not search for new models, alter the frozen train–calibration–test partitions, or use the test set for selection.

### Fixed manuscript models

- **Dry Bean:** `R1_full_native + RBF-SVM`.
- **INIAP Rice:** `R4_complete_log_area + RBF-SVM`.

The Dry Bean model is retained as the prespecified primary model from the original frozen analysis. The representation reanalysis from Notebook 12 is confirmatory and is reported separately. The INIAP Rice primary model remains the complete invariant representation plus log-transformed pixel area.

### What this notebook produces

- A final analysis-freeze protocol.
- Conventional champion metrics with group-bootstrap intervals.
- Corrected representation contrasts and exact McNemar results.
- Rice equivalence results.
- Calibration diagnostics before and after temperature scaling.
- Selective-classification operating points and risk–coverage curves.
- LAC, randomized APS, and deterministic APS diagnostics.
- Confusion matrices and classwise metrics.
- Controlled R1 size–shape importance for both crops.
- Controlled image-space scale sensitivity.
- Duplicate-leakage audits.
- Main-manuscript and supplementary tables in CSV, Markdown, and LaTeX.
- Main and supplementary figures in PNG and PDF.
- A numerical claim register.
- A reviewer-response evidence map.
- SHA-256 integrity manifests and a final output ZIP.

### Interpretation boundary

All scale-dependent variables are image-space descriptors:

- linear dimensions and perimeter are measured in pixels;
- area variables are measured in squared pixels;
- ratios and shape factors are dimensionless.

No conversion to millimetres is assumed, and no absolute size values are compared directly between datasets.

In [ ]:
# ============================================================
# 1. Dependencies and global configuration
# ============================================================
import importlib
import importlib.metadata
import subprocess
import sys

REQUIRED_PACKAGES = {
    "pyarrow": "pyarrow",
    "tabulate": "tabulate",
}

to_install = []

for module_name, package_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(module_name) is None:
        to_install.append(package_name)

if to_install:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *to_install]
    )

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import math
import os
import platform
import shutil
import time
import zipfile
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore")

SOURCE_ZIP_NAME = (
    "NOTEBOOK12_REVIEWER_DRIVEN_REANALYSIS_OUTPUTS_V2.zip"
)

FINAL_CHAMPIONS = {
    "dry_bean": "R1_full_native",
    "iniap_rice": "R4_complete_log_area",
}

CERTAINTY_SCORES = {
    "dry_bean": "max_probability",
    "iniap_rice": "top2_margin",
}

SELECTIVE_TARGET_COVERAGE = 0.90
CONFORMAL_LEVELS = [0.90, 0.95]
CONFORMAL_METHODS = [
    "LAC",
    "APS_randomized",
    "APS_deterministic",
]

N_BOOTSTRAP = 2000
CONFIDENCE_LEVEL = 0.95
N_RELIABILITY_BINS = 15
FINAL_SEED = 2026081301
CONFORMAL_SEED = 2026081205

LOCAL_ROOT = Path(
    "/content/Grain_AI_MDPI_notebook13"
)

INPUT_DIR = LOCAL_ROOT / "00_input"
EXTRACT_DIR = LOCAL_ROOT / "01_extracted_notebook12"
OUTPUT_ROOT = LOCAL_ROOT / "02_outputs"

PROTOCOL_DIR = OUTPUT_ROOT / "03_protocol"
RESULTS_DIR = OUTPUT_ROOT / "07_results" / "13_FINAL_FREEZE"
FIGURES_MAIN_DIR = OUTPUT_ROOT / "08_figures" / "main"
FIGURES_SUPP_DIR = OUTPUT_ROOT / "08_figures" / "supplementary"
TABLES_MAIN_DIR = OUTPUT_ROOT / "09_tables" / "main"
TABLES_SUPP_DIR = OUTPUT_ROOT / "09_tables" / "supplementary"
CLAIMS_DIR = OUTPUT_ROOT / "10_claim_register"

for directory in [
    INPUT_DIR,
    EXTRACT_DIR,
    PROTOCOL_DIR,
    RESULTS_DIR,
    FIGURES_MAIN_DIR,
    FIGURES_SUPP_DIR,
    TABLES_MAIN_DIR,
    TABLES_SUPP_DIR,
    CLAIMS_DIR,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

print("Python:", sys.version.split()[0])
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print(
    "scikit-learn:",
    importlib.metadata.version("scikit-learn"),
)
print(
    "pyarrow:",
    importlib.metadata.version("pyarrow"),
)
print("Platform:", platform.platform())
print("CPU count:", os.cpu_count())
print("Bootstrap repetitions:", N_BOOTSTRAP)

In [ ]:
# ============================================================
# 2. Robust Google Drive mounting and source ZIP discovery
# ============================================================
def resolve_colab_drive_mount():
    try:
        from google.colab import drive, files
    except ModuleNotFoundError as exc:
        raise RuntimeError(
            "This notebook must be executed in a hosted Google "
            "Colab runtime."
        ) from exc

    existing_candidates = [
        Path("/content/drive"),
        Path("/content/drive_nb12"),
        Path("/content/drive_nb13"),
    ]

    for candidate in existing_candidates:
        if (
            candidate.exists()
            and (
                candidate
                / "MyDrive"
            ).exists()
        ):
            print(
                "Using existing Drive mount:",
                candidate,
            )
            return candidate, files

    mount_candidates = [
        Path("/content/drive"),
        Path("/content/drive_nb13"),
    ]

    last_error = None

    for mount_point in mount_candidates:
        try:
            if mount_point.exists():
                existing_items = list(
                    mount_point.iterdir()
                )

                if existing_items:
                    continue
            else:
                mount_point.mkdir(
                    parents=True,
                    exist_ok=True,
                )

            drive.mount(
                str(
                    mount_point
                ),
                force_remount=False,
            )

            if (
                mount_point
                / "MyDrive"
            ).exists():
                print(
                    "Mounted Google Drive at:",
                    mount_point,
                )
                return mount_point, files

        except Exception as exc:
            last_error = exc

    raise RuntimeError(
        "Google Drive could not be mounted at an empty mount point."
    ) from last_error

MOUNT_ROOT, colab_files = resolve_colab_drive_mount()

DRIVE_PROJECT_ROOT = (
    MOUNT_ROOT
    / "MyDrive"
    / "Grain_Robustness_Q1"
)

DRIVE_PROJECT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

def find_source_zip():
    exact_candidates = [
        DRIVE_PROJECT_ROOT
        / SOURCE_ZIP_NAME,
        Path("/content")
        / SOURCE_ZIP_NAME,
    ]

    for candidate in exact_candidates:
        if candidate.exists():
            return candidate

    matches = sorted(
        (
            MOUNT_ROOT
            / "MyDrive"
        ).rglob(
            SOURCE_ZIP_NAME
        )
    )

    if matches:
        return max(
            matches,
            key=lambda path: path.stat().st_mtime,
        )

    print(
        "Upload the Notebook 12 FULL output ZIP:",
        SOURCE_ZIP_NAME,
    )

    uploaded = colab_files.upload()

    if SOURCE_ZIP_NAME not in uploaded:
        raise FileNotFoundError(
            SOURCE_ZIP_NAME
        )

    destination = (
        INPUT_DIR
        / SOURCE_ZIP_NAME
    )

    destination.write_bytes(
        uploaded[
            SOURCE_ZIP_NAME
        ]
    )

    return destination

SOURCE_ZIP_PATH = find_source_zip()

print("Source ZIP:", SOURCE_ZIP_PATH)
print(
    "Source ZIP size (MB):",
    round(
        SOURCE_ZIP_PATH.stat().st_size
        / 1e6,
        2,
    ),
)

In [ ]:
# ============================================================
# 3. ZIP extraction and integrity validation
# ============================================================
def sha256_file(path: Path):
    digest = hashlib.sha256()

    with path.open("rb") as stream:
        for block in iter(
            lambda: stream.read(
                1024
                * 1024
            ),
            b"",
        ):
            digest.update(
                block
            )

    return digest.hexdigest()

with zipfile.ZipFile(
    SOURCE_ZIP_PATH,
    "r",
) as archive:
    corrupt_member = archive.testzip()

    if corrupt_member is not None:
        raise RuntimeError(
            f"Corrupt ZIP member: {corrupt_member}"
        )

if EXTRACT_DIR.exists():
    shutil.rmtree(
        EXTRACT_DIR
    )

EXTRACT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

with zipfile.ZipFile(
    SOURCE_ZIP_PATH,
    "r",
) as archive:
    archive.extractall(
        EXTRACT_DIR
    )

def find_one(root: Path, filename: str):
    matches = sorted(
        root.rglob(
            filename
        )
    )

    if not matches:
        raise FileNotFoundError(
            f"{filename} was not found under {root}."
        )

    return matches[0]

source_validation_path = find_one(
    EXTRACT_DIR,
    "03_final_reviewer_reanalysis_validation.csv",
)

source_manifest_path = find_one(
    EXTRACT_DIR,
    "04_output_integrity_manifest.csv",
)

source_protocol_path = find_one(
    EXTRACT_DIR,
    "reviewer_reanalysis_protocol_v2.json",
)

source_validation = pd.read_csv(
    source_validation_path
)

source_manifest = pd.read_csv(
    source_manifest_path
)

source_protocol = json.loads(
    source_protocol_path.read_text(
        encoding="utf-8"
    )
)

if not source_validation[
    "passed"
].astype(bool).all():
    display(source_validation)

    raise RuntimeError(
        "Notebook 12 did not pass all final validations."
    )

if source_protocol.get(
    "run_mode"
) != "FULL":
    raise RuntimeError(
        "Notebook 13 requires Notebook 12 outputs produced in FULL mode."
    )

manifest_validation_rows = []

for _, manifest_row in source_manifest.iterrows():
    relative_path = Path(
        str(
            manifest_row[
                "relative_path"
            ]
        )
    )

    candidate = (
        EXTRACT_DIR
        / relative_path
    )

    exists = candidate.exists()

    size_matches = (
        exists
        and candidate.stat().st_size
        == int(
            manifest_row[
                "size_bytes"
            ]
        )
    )

    hash_matches = (
        exists
        and sha256_file(
            candidate
        )
        == str(
            manifest_row[
                "sha256"
            ]
        )
    )

    manifest_validation_rows.append(
        {
            "relative_path": str(
                relative_path
            ),
            "exists": exists,
            "size_matches": size_matches,
            "hash_matches": hash_matches,
            "passed": bool(
                exists
                and size_matches
                and hash_matches
            ),
        }
    )

source_integrity_validation = pd.DataFrame(
    manifest_validation_rows
)

if not source_integrity_validation[
    "passed"
].all():
    display(
        source_integrity_validation.loc[
            ~source_integrity_validation[
                "passed"
            ]
        ]
    )

    raise RuntimeError(
        "The Notebook 12 integrity manifest did not validate."
    )

print(
    "Notebook 12 validation checks:",
    len(
        source_validation
    ),
)

print(
    "Notebook 12 manifest files validated:",
    len(
        source_integrity_validation
    ),
)

print(
    "Notebook 12 source integrity: PASSED"
)

In [ ]:
# ============================================================
# 4. Source table and prediction loading
# ============================================================
SOURCE_TABLE_DIR = (
    EXTRACT_DIR
    / "09_tables"
    / "12_REVIEWER_REANALYSIS"
)

SOURCE_RESULTS_DIR = (
    EXTRACT_DIR
    / "07_results"
    / "12_REVIEWER_REANALYSIS"
)

TABLE_FILES = {
    "representation_audit": (
        "Table_01_representation_audit.csv"
    ),
    "all_candidates": (
        "Table_02_training_only_rbf_selection_all_candidates.csv"
    ),
    "selected_candidates": (
        "Table_03_training_selected_rbf_candidates.csv"
    ),
    "timing": (
        "Table_04_computational_timing.csv"
    ),
    "representation_metrics": (
        "Table_05_corrected_representation_test_metrics.csv"
    ),
    "paired_contrasts": (
        "Table_06_corrected_paired_representation_contrasts.csv"
    ),
    "mcnemar": (
        "Table_07_exact_mcnemar_discordant_counts.csv"
    ),
    "rice_equivalence": (
        "Table_08_rice_bootstrap_TOST_equivalence.csv"
    ),
    "controlled_importance": (
        "Table_09_controlled_R1_size_shape_permutation.csv"
    ),
    "controlled_scale": (
        "Table_10_controlled_R1_scale_bias.csv"
    ),
    "nb12_champions": (
        "Table_11_revised_training_selected_champions.csv"
    ),
    "nb12_calibration": (
        "Table_12_calibration_audit.csv"
    ),
    "nb12_selective": (
        "Table_13_selective_results_row_and_group_units.csv"
    ),
    "nb12_conformal": (
        "Table_14_revised_conformal_empty_set_diagnostics.csv"
    ),
    "nb12_conformal_classwise": (
        "Table_15_revised_conformal_classwise.csv"
    ),
    "nb12_conformal_quantiles": (
        "Table_16_revised_conformal_quantiles.csv"
    ),
    "naive_leakage": (
        "Table_17_repeated_naive_row_split_leakage_audit.csv"
    ),
    "fixed_leakage": (
        "Table_18_fixed_test_duplicate_contamination_audit.csv"
    ),
    "classwise_nb12": (
        "Table_20_complete_classwise_metrics.csv"
    ),
}

source_tables = {
    key: pd.read_csv(
        SOURCE_TABLE_DIR
        / filename
    )
    for key, filename in TABLE_FILES.items()
}

model_metadata = pd.read_csv(
    SOURCE_RESULTS_DIR
    / "01_model_metadata.csv"
)

def class_order_for(
    dataset_id,
    representation,
):
    row = model_metadata.loc[
        (
            model_metadata[
                "dataset_id"
            ] == dataset_id
        )
        & (
            model_metadata[
                "representation"
            ] == representation
        )
    ]

    if len(
        row
    ) != 1:
        raise RuntimeError(
            "Champion metadata row was not unique."
        )

    return [
        value.strip()
        for value in row.iloc[0][
            "class_order"
        ].split("|")
    ]

def load_prediction_frame(
    dataset_id,
    representation,
    partition,
):
    path = (
        EXTRACT_DIR
        / "06_predictions"
        / dataset_id
        / representation
        / (
            f"rbf_svm_{partition}_"
            "predictions.parquet"
        )
    )

    if not path.exists():
        raise FileNotFoundError(
            path
        )

    frame = pd.read_parquet(
        path
    )

    for column in [
        "dataset_id",
        "record_id",
        "duplicate_group_id",
        "class_label",
    ]:
        frame[
            column
        ] = frame[
            column
        ].astype(str)

    return frame

prediction_frames = {}

for dataset_id, representation in FINAL_CHAMPIONS.items():
    for partition in [
        "calibration",
        "test",
    ]:
        prediction_frames[
            (
                dataset_id,
                partition,
            )
        ] = load_prediction_frame(
            dataset_id,
            representation,
            partition,
        )

frozen_champion_table = pd.DataFrame(
    [
        {
            "dataset_id": dataset_id,
            "frozen_primary_representation": representation,
            "model_family": "RBF-SVM",
            "selection_status": (
                "prespecified primary model; "
                "not reselected after reviewer reanalysis"
            ),
            "measurement_domain": (
                "image-space pixel descriptors"
            ),
        }
        for dataset_id, representation
        in FINAL_CHAMPIONS.items()
    ]
)

display(frozen_champion_table)

for key, frame in prediction_frames.items():
    print(
        key,
        frame.shape,
    )

In [ ]:
# ============================================================
# 5. Probability parsing and metric functions
# ============================================================
def sorted_probability_columns(
    frame,
    prefix,
):
    columns = [
        column
        for column in frame.columns
        if column.startswith(
            prefix
        )
    ]

    return sorted(
        columns,
        key=lambda column: int(
            column.rsplit(
                "_",
                1,
            )[
                -1
            ]
        ),
    )

def probability_matrix(
    frame,
    status,
):
    if status == "native":
        prefix = "prob_native_"
    elif status == "temperature_scaled":
        prefix = (
            "prob_temperature_scaled_"
        )
    else:
        raise ValueError(
            status
        )

    columns = sorted_probability_columns(
        frame,
        prefix,
    )

    if not columns:
        raise RuntimeError(
            f"No probability columns with prefix {prefix}."
        )

    probabilities = frame[
        columns
    ].to_numpy(
        dtype=float
    )

    row_sums = probabilities.sum(
        axis=1,
        keepdims=True,
    )

    if (
        not np.isfinite(
            probabilities
        ).all()
        or (
            row_sums
            <= 0
        ).any()
    ):
        raise RuntimeError(
            "Invalid probability matrix."
        )

    return (
        probabilities
        / row_sums
    )

def probability_labels(
    probabilities,
    class_order,
):
    class_order = np.asarray(
        class_order,
        dtype=str,
    )

    return class_order[
        np.asarray(
            probabilities,
            dtype=float,
        ).argmax(
            axis=1
        )
    ]

def multiclass_brier(
    y_true,
    probabilities,
    class_order,
):
    y_true = np.asarray(
        y_true,
        dtype=str,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    class_to_index = {
        class_label: index
        for index, class_label
        in enumerate(
            class_order
        )
    }

    indicator = np.zeros_like(
        probabilities
    )

    for row_index, class_label in enumerate(
        y_true
    ):
        indicator[
            row_index,
            class_to_index[
                class_label
            ],
        ] = 1.0

    return float(
        np.mean(
            np.sum(
                (
                    probabilities
                    - indicator
                )
                ** 2,
                axis=1,
            )
        )
    )

def metric_bundle(
    y_true,
    probabilities,
    class_order,
):
    labels = probability_labels(
        probabilities,
        class_order,
    )

    return {
        "accuracy": float(
            accuracy_score(
                y_true,
                labels,
            )
        ),
        "balanced_accuracy": float(
            balanced_accuracy_score(
                y_true,
                labels,
            )
        ),
        "macro_f1": float(
            f1_score(
                y_true,
                labels,
                labels=class_order,
                average="macro",
                zero_division=0,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                y_true,
                labels,
            )
        ),
        "log_loss": float(
            log_loss(
                y_true,
                probabilities,
                labels=class_order,
            )
        ),
        "brier_multiclass": (
            multiclass_brier(
                y_true,
                probabilities,
                class_order,
            )
        ),
    }

def percentile_interval(
    values,
    confidence_level=0.95,
):
    values = np.asarray(
        values,
        dtype=float,
    )

    alpha = (
        1.0
        - confidence_level
    ) / 2.0

    return (
        float(
            np.quantile(
                values,
                alpha,
            )
        ),
        float(
            np.quantile(
                values,
                1.0
                - alpha,
            )
        ),
    )

def aggregate_group_predictions(
    frame,
    probabilities,
):
    probability_columns = [
        f"_p_{index}"
        for index in range(
            probabilities.shape[
                1
            ]
        )
    ]

    working = frame[
        [
            "dataset_id",
            "record_id",
            "duplicate_group_id",
            "class_label",
        ]
    ].copy()

    working[
        probability_columns
    ] = probabilities

    label_counts = (
        working.groupby(
            "duplicate_group_id"
        )[
            "class_label"
        ]
        .nunique()
    )

    if (
        label_counts
        > 1
    ).any():
        raise RuntimeError(
            "A duplicate group contains multiple class labels."
        )

    probability_ranges = (
        working.groupby(
            "duplicate_group_id"
        )[
            probability_columns
        ]
        .max()
        - working.groupby(
            "duplicate_group_id"
        )[
            probability_columns
        ]
        .min()
    )

    if (
        probability_ranges.to_numpy()
        > 1e-12
    ).any():
        raise RuntimeError(
            "Duplicate rows do not have identical probabilities."
        )

    group_frame = (
        working.groupby(
            "duplicate_group_id",
            as_index=False,
            sort=True,
        )
        .first()
    )

    group_probabilities = group_frame[
        probability_columns
    ].to_numpy(
        dtype=float
    )

    group_frame = group_frame.drop(
        columns=probability_columns
    )

    return (
        group_frame,
        group_probabilities,
    )

In [ ]:
# ============================================================
# 6. Frozen champion performance with group-bootstrap intervals
# ============================================================
def stratified_group_bootstrap_indices(
    frame,
    n_bootstrap,
    seed,
):
    group_labels = (
        frame[
            [
                "duplicate_group_id",
                "class_label",
            ]
        ]
        .drop_duplicates()
        .sort_values(
            [
                "class_label",
                "duplicate_group_id",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    group_to_indices = {
        group_id: frame.index[
            frame[
                "duplicate_group_id"
            ] == group_id
        ].to_numpy(
            dtype=int
        )
        for group_id in group_labels[
            "duplicate_group_id"
        ]
    }

    groups_by_class = {
        class_label: group_labels.loc[
            group_labels[
                "class_label"
            ] == class_label,
            "duplicate_group_id",
        ].to_numpy()
        for class_label in sorted(
            group_labels[
                "class_label"
            ].unique()
        )
    }

    rng = np.random.default_rng(
        seed
    )

    bootstrap_indices = []

    for _ in range(
        n_bootstrap
    ):
        sampled_indices = []

        for _, class_groups in groups_by_class.items():
            sampled_groups = rng.choice(
                class_groups,
                size=len(
                    class_groups
                ),
                replace=True,
            )

            for group_id in sampled_groups:
                sampled_indices.extend(
                    group_to_indices[
                        group_id
                    ].tolist()
                )

        bootstrap_indices.append(
            np.asarray(
                sampled_indices,
                dtype=int,
            )
        )

    return bootstrap_indices

champion_metric_rows = []
champion_bootstrap_rows = []

for dataset_index, (
    dataset_id,
    representation,
) in enumerate(
    FINAL_CHAMPIONS.items()
):
    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ].reset_index(
        drop=True
    )

    class_order = class_order_for(
        dataset_id,
        representation,
    )

    probabilities = probability_matrix(
        test_frame,
        "native",
    )

    y_true = test_frame[
        "class_label"
    ].to_numpy(
        dtype=str
    )

    row_metrics = metric_bundle(
        y_true,
        probabilities,
        class_order,
    )

    group_frame, group_probabilities = (
        aggregate_group_predictions(
            test_frame,
            probabilities,
        )
    )

    group_metrics = metric_bundle(
        group_frame[
            "class_label"
        ].to_numpy(
            dtype=str
        ),
        group_probabilities,
        class_order,
    )

    for evaluation_unit, metrics, n_units in [
        (
            "row",
            row_metrics,
            len(
                test_frame
            ),
        ),
        (
            "duplicate_group_id",
            group_metrics,
            len(
                group_frame
            ),
        ),
    ]:
        champion_metric_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": representation,
                "evaluation_unit": evaluation_unit,
                "n_units": n_units,
                **metrics,
            }
        )

    bootstrap_indices = (
        stratified_group_bootstrap_indices(
            test_frame,
            N_BOOTSTRAP,
            FINAL_SEED
            + dataset_index,
        )
    )

    metric_names = [
        "accuracy",
        "balanced_accuracy",
        "macro_f1",
        "mcc",
        "log_loss",
        "brier_multiclass",
    ]

    bootstrap_values = {
        metric_name: np.empty(
            N_BOOTSTRAP,
            dtype=float,
        )
        for metric_name in metric_names
    }

    for bootstrap_index, sampled_indices in enumerate(
        bootstrap_indices
    ):
        metrics = metric_bundle(
            y_true[
                sampled_indices
            ],
            probabilities[
                sampled_indices
            ],
            class_order,
        )

        for metric_name in metric_names:
            bootstrap_values[
                metric_name
            ][
                bootstrap_index
            ] = metrics[
                metric_name
            ]

    for metric_name in metric_names:
        ci_low, ci_high = percentile_interval(
            bootstrap_values[
                metric_name
            ],
            CONFIDENCE_LEVEL,
        )

        champion_bootstrap_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": representation,
                "evaluation_unit": (
                    "row metrics with duplicate-group bootstrap"
                ),
                "metric": metric_name,
                "estimate": row_metrics[
                    metric_name
                ],
                "ci_low": ci_low,
                "ci_high": ci_high,
                "confidence_level": CONFIDENCE_LEVEL,
                "n_bootstrap": N_BOOTSTRAP,
            }
        )

champion_metrics = pd.DataFrame(
    champion_metric_rows
)

champion_metric_intervals = pd.DataFrame(
    champion_bootstrap_rows
)

display(champion_metrics)
display(champion_metric_intervals)

In [ ]:
# ============================================================
# 7. Calibration audit and reliability diagrams
# ============================================================
def expected_calibration_error(
    y_true,
    probabilities,
    class_order,
    n_bins=15,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    y_true = np.asarray(
        y_true,
        dtype=str,
    )

    predicted = probability_labels(
        probabilities,
        class_order,
    )

    confidence = probabilities.max(
        axis=1
    )

    correct = (
        predicted
        == y_true
    ).astype(float)

    edges = np.linspace(
        0.0,
        1.0,
        n_bins
        + 1,
    )

    ece = 0.0
    maximum_gap = 0.0
    rows = []

    for bin_index in range(
        n_bins
    ):
        lower = edges[
            bin_index
        ]

        upper = edges[
            bin_index
            + 1
        ]

        if bin_index == (
            n_bins
            - 1
        ):
            mask = (
                confidence
                >= lower
            ) & (
                confidence
                <= upper
            )
        else:
            mask = (
                confidence
                >= lower
            ) & (
                confidence
                < upper
            )

        count = int(
            mask.sum()
        )

        if count == 0:
            continue

        empirical_accuracy = float(
            correct[
                mask
            ].mean()
        )

        mean_confidence = float(
            confidence[
                mask
            ].mean()
        )

        gap = abs(
            empirical_accuracy
            - mean_confidence
        )

        ece += (
            count
            / len(
                confidence
            )
        ) * gap

        maximum_gap = max(
            maximum_gap,
            gap,
        )

        rows.append(
            {
                "bin_index": bin_index,
                "lower": lower,
                "upper": upper,
                "count": count,
                "empirical_accuracy": empirical_accuracy,
                "mean_confidence": mean_confidence,
                "absolute_gap": gap,
            }
        )

    return (
        float(
            ece
        ),
        float(
            maximum_gap
        ),
        pd.DataFrame(
            rows
        ),
    )

calibration_rows = []

for dataset_id, representation in FINAL_CHAMPIONS.items():
    metadata_row = model_metadata.loc[
        (
            model_metadata[
                "dataset_id"
            ] == dataset_id
        )
        & (
            model_metadata[
                "representation"
            ] == representation
        )
    ].iloc[0]

    class_order = class_order_for(
        dataset_id,
        representation,
    )

    for partition in [
        "calibration",
        "test",
    ]:
        frame = prediction_frames[
            (
                dataset_id,
                partition,
            )
        ]

        y_true = frame[
            "class_label"
        ].to_numpy(
            dtype=str
        )

        bin_tables = {}

        for probability_status in [
            "native",
            "temperature_scaled",
        ]:
            probabilities = probability_matrix(
                frame,
                probability_status,
            )

            metrics = metric_bundle(
                y_true,
                probabilities,
                class_order,
            )

            ece, maximum_error, bins = (
                expected_calibration_error(
                    y_true,
                    probabilities,
                    class_order,
                    n_bins=N_RELIABILITY_BINS,
                )
            )

            bin_tables[
                probability_status
            ] = bins

            bins.to_csv(
                RESULTS_DIR
                / (
                    f"reliability_bins_{dataset_id}_"
                    f"{partition}_{probability_status}.csv"
                ),
                index=False,
                encoding="utf-8-sig",
            )

            calibration_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": representation,
                    "partition": partition,
                    "probability_status": probability_status,
                    "temperature": float(
                        metadata_row[
                            "temperature"
                        ]
                    ),
                    "log_loss": metrics[
                        "log_loss"
                    ],
                    "brier_multiclass": metrics[
                        "brier_multiclass"
                    ],
                    "ece_equal_width": ece,
                    "maximum_calibration_error": maximum_error,
                    "n_bins": N_RELIABILITY_BINS,
                }
            )

        figure, axis = plt.subplots(
            figsize=(
                7,
                6,
            )
        )

        axis.plot(
            [
                0.0,
                1.0,
            ],
            [
                0.0,
                1.0,
            ],
            linestyle="--",
            linewidth=1,
            label="Perfect calibration",
        )

        for probability_status, label in [
            (
                "native",
                "Native",
            ),
            (
                "temperature_scaled",
                "Temperature-scaled",
            ),
        ]:
            bins = bin_tables[
                probability_status
            ]

            axis.plot(
                bins[
                    "mean_confidence"
                ],
                bins[
                    "empirical_accuracy"
                ],
                marker="o",
                label=label,
            )

        axis.set_xlabel(
            "Mean predicted confidence"
        )

        axis.set_ylabel(
            "Empirical accuracy"
        )

        axis.set_title(
            (
                f"Reliability diagram — "
                f"{dataset_id.replace('_', ' ').title()} — "
                f"{partition}"
            )
        )

        axis.grid(
            True,
            alpha=0.3,
        )

        axis.legend()
        figure.tight_layout()

        for extension in [
            "png",
            "pdf",
        ]:
            figure.savefig(
                FIGURES_SUPP_DIR
                / (
                    f"Figure_S_reliability_{dataset_id}_"
                    f"{partition}.{extension}"
                ),
                dpi=300
                if extension
                == "png"
                else None,
                bbox_inches="tight",
            )

        plt.show()

calibration_audit = pd.DataFrame(
    calibration_rows
)

display(calibration_audit)

In [ ]:
# ============================================================
# 8. Selective classification and risk–coverage curves
# ============================================================
def certainty_values(
    probabilities,
    score_name,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    sorted_probabilities = np.sort(
        probabilities,
        axis=1,
    )

    if score_name == "max_probability":
        return sorted_probabilities[
            :,
            -1
        ]

    if score_name == "top2_margin":
        return (
            sorted_probabilities[
                :,
                -1
            ]
            - sorted_probabilities[
                :,
                -2
            ]
        )

    raise ValueError(
        score_name
    )

def threshold_for_target_coverage(
    scores,
    target_coverage,
):
    scores = np.sort(
        np.asarray(
            scores,
            dtype=float,
        )
    )

    retained_count = int(
        math.ceil(
            target_coverage
            * len(
                scores
            )
        )
    )

    index = max(
        0,
        len(
            scores
        )
        - retained_count,
    )

    return float(
        scores[
            index
        ]
    )

selective_rows = []
risk_coverage_rows = []

for dataset_id, representation in FINAL_CHAMPIONS.items():
    class_order = class_order_for(
        dataset_id,
        representation,
    )

    score_name = CERTAINTY_SCORES[
        dataset_id
    ]

    calibration_frame = prediction_frames[
        (
            dataset_id,
            "calibration",
        )
    ].copy()

    calibration_probabilities = probability_matrix(
        calibration_frame,
        "temperature_scaled",
    )

    calibration_frame[
        "certainty"
    ] = certainty_values(
        calibration_probabilities,
        score_name,
    )

    calibration_group_scores = (
        calibration_frame.groupby(
            "duplicate_group_id",
            as_index=False,
            sort=True,
        )
        .agg(
            certainty=(
                "certainty",
                "min",
            ),
            class_label=(
                "class_label",
                "first",
            ),
        )
    )

    primary_threshold = threshold_for_target_coverage(
        calibration_group_scores[
            "certainty"
        ],
        SELECTIVE_TARGET_COVERAGE,
    )

    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ].copy()

    test_native = probability_matrix(
        test_frame,
        "native",
    )

    test_scaled = probability_matrix(
        test_frame,
        "temperature_scaled",
    )

    test_frame[
        "certainty"
    ] = certainty_values(
        test_scaled,
        score_name,
    )

    test_frame[
        "prediction"
    ] = probability_labels(
        test_native,
        class_order,
    )

    for evaluation_unit in [
        "row",
        "duplicate_group_id",
    ]:
        if evaluation_unit == "row":
            evaluation = test_frame.copy()
        else:
            evaluation = (
                test_frame.groupby(
                    "duplicate_group_id",
                    as_index=False,
                    sort=True,
                )
                .agg(
                    class_label=(
                        "class_label",
                        "first",
                    ),
                    prediction=(
                        "prediction",
                        "first",
                    ),
                    certainty=(
                        "certainty",
                        "min",
                    ),
                )
            )

        retained = (
            evaluation[
                "certainty"
            ].to_numpy(
                dtype=float
            )
            >= primary_threshold
        )

        y_true = evaluation[
            "class_label"
        ].to_numpy(
            dtype=str
        )

        y_pred = evaluation[
            "prediction"
        ].to_numpy(
            dtype=str
        )

        baseline_risk = (
            1.0
            - accuracy_score(
                y_true,
                y_pred,
            )
        )

        if retained.any():
            selective_accuracy = float(
                accuracy_score(
                    y_true[
                        retained
                    ],
                    y_pred[
                        retained
                    ],
                )
            )

            selective_macro_f1 = float(
                f1_score(
                    y_true[
                        retained
                    ],
                    y_pred[
                        retained
                    ],
                    labels=class_order,
                    average="macro",
                    zero_division=0,
                )
            )
        else:
            selective_accuracy = np.nan
            selective_macro_f1 = np.nan

        selective_risk = (
            1.0
            - selective_accuracy
        )

        selective_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": representation,
                "certainty_score": score_name,
                "calibration_unit": "duplicate_group_id",
                "evaluation_unit": evaluation_unit,
                "target_calibration_coverage": (
                    SELECTIVE_TARGET_COVERAGE
                ),
                "certainty_threshold": primary_threshold,
                "n_total": len(
                    evaluation
                ),
                "n_retained": int(
                    retained.sum()
                ),
                "coverage": float(
                    retained.mean()
                ),
                "selective_accuracy": selective_accuracy,
                "selective_macro_f1": selective_macro_f1,
                "selective_risk": selective_risk,
                "baseline_risk": baseline_risk,
                "risk_reduction": (
                    baseline_risk
                    - selective_risk
                ),
            }
        )

    for target_coverage in np.linspace(
        0.50,
        1.00,
        26,
    ):
        threshold = threshold_for_target_coverage(
            calibration_group_scores[
                "certainty"
            ],
            float(
                target_coverage
            ),
        )

        retained = (
            test_frame[
                "certainty"
            ].to_numpy(
                dtype=float
            )
            >= threshold
        )

        y_true = test_frame[
            "class_label"
        ].to_numpy(
            dtype=str
        )

        y_pred = test_frame[
            "prediction"
        ].to_numpy(
            dtype=str
        )

        if retained.any():
            risk = (
                1.0
                - accuracy_score(
                    y_true[
                        retained
                    ],
                    y_pred[
                        retained
                    ],
                )
            )
        else:
            risk = np.nan

        risk_coverage_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": representation,
                "certainty_score": score_name,
                "target_calibration_coverage": float(
                    target_coverage
                ),
                "threshold": threshold,
                "test_coverage": float(
                    retained.mean()
                ),
                "test_risk": float(
                    risk
                ),
            }
        )

selective_results = pd.DataFrame(
    selective_rows
)

risk_coverage_results = pd.DataFrame(
    risk_coverage_rows
)

figure, axis = plt.subplots(
    figsize=(
        8,
        6,
    )
)

for dataset_id in FINAL_CHAMPIONS:
    subset = risk_coverage_results.loc[
        risk_coverage_results[
            "dataset_id"
        ] == dataset_id
    ].sort_values(
        "test_coverage"
    )

    axis.plot(
        subset[
            "test_coverage"
        ],
        subset[
            "test_risk"
        ],
        marker="o",
        markersize=3,
        label=dataset_id.replace(
            "_",
            " ",
        ).title(),
    )

axis.set_xlabel(
    "Empirical test coverage"
)

axis.set_ylabel(
    "Selective risk"
)

axis.set_title(
    "Risk–coverage curves of the frozen primary models"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()
figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_4_risk_coverage_curves.{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

display(selective_results)

In [ ]:
# ============================================================
# 9. Conformal diagnostics for the frozen primary models
# ============================================================
def stable_uniform(
    seed,
    dataset_id,
    group_id,
    class_label,
    method,
):
    token = (
        f"{seed}|{dataset_id}|{group_id}|"
        f"{class_label}|{method}"
    )

    digest = hashlib.sha256(
        token.encode(
            "utf-8"
        )
    ).digest()

    integer = int.from_bytes(
        digest[
            :8
        ],
        byteorder="big",
        signed=False,
    )

    return (
        integer
        + 0.5
    ) / (
        2
        ** 64
    )

def conformal_candidate_scores(
    probabilities,
    class_order,
    dataset_id,
    group_ids,
    method,
):
    probabilities = np.asarray(
        probabilities,
        dtype=float,
    )

    class_order = np.asarray(
        class_order,
        dtype=str,
    )

    scores = np.zeros_like(
        probabilities
    )

    for row_index in range(
        len(
            probabilities
        )
    ):
        row = probabilities[
            row_index
        ]

        order = np.argsort(
            -row,
            kind="mergesort",
        )

        cumulative_before = 0.0

        for _, class_index in enumerate(
            order
        ):
            probability = row[
                class_index
            ]

            if method == "LAC":
                score = (
                    1.0
                    - probability
                )

            elif method == "APS_randomized":
                uniform = stable_uniform(
                    CONFORMAL_SEED,
                    dataset_id,
                    group_ids[
                        row_index
                    ],
                    class_order[
                        class_index
                    ],
                    method,
                )

                score = (
                    cumulative_before
                    + uniform
                    * probability
                )

            elif method == "APS_deterministic":
                score = (
                    cumulative_before
                    + probability
                )

            else:
                raise ValueError(
                    method
                )

            scores[
                row_index,
                class_index,
            ] = score

            cumulative_before += probability

    return scores

def finite_sample_quantile(
    values,
    nominal_coverage,
):
    values = np.sort(
        np.asarray(
            values,
            dtype=float,
        )
    )

    order_index = int(
        math.ceil(
            (
                len(
                    values
                )
                + 1
            )
            * nominal_coverage
        )
    )

    order_index = min(
        max(
            order_index,
            1,
        ),
        len(
            values
        ),
    )

    return (
        float(
            values[
                order_index
                - 1
            ]
        ),
        order_index,
    )

conformal_rows = []
conformal_class_rows = []
conformal_quantile_rows = []

for dataset_id, representation in FINAL_CHAMPIONS.items():
    class_order = class_order_for(
        dataset_id,
        representation,
    )

    class_to_index = {
        class_label: index
        for index, class_label
        in enumerate(
            class_order
        )
    }

    calibration_frame = prediction_frames[
        (
            dataset_id,
            "calibration",
        )
    ].copy()

    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ].copy()

    calibration_probabilities = probability_matrix(
        calibration_frame,
        "temperature_scaled",
    )

    test_probabilities = probability_matrix(
        test_frame,
        "temperature_scaled",
    )

    calibration_true_indices = np.asarray(
        [
            class_to_index[
                class_label
            ]
            for class_label in calibration_frame[
                "class_label"
            ]
        ],
        dtype=int,
    )

    test_true_indices = np.asarray(
        [
            class_to_index[
                class_label
            ]
            for class_label in test_frame[
                "class_label"
            ]
        ],
        dtype=int,
    )

    for method in CONFORMAL_METHODS:
        calibration_scores = (
            conformal_candidate_scores(
                calibration_probabilities,
                class_order,
                dataset_id,
                calibration_frame[
                    "duplicate_group_id"
                ].to_numpy(
                    dtype=str
                ),
                method,
            )
        )

        test_scores = (
            conformal_candidate_scores(
                test_probabilities,
                class_order,
                dataset_id,
                test_frame[
                    "duplicate_group_id"
                ].to_numpy(
                    dtype=str
                ),
                method,
            )
        )

        calibration_true_scores = calibration_scores[
            np.arange(
                len(
                    calibration_frame
                )
            ),
            calibration_true_indices,
        ]

        calibration_score_frame = pd.DataFrame(
            {
                "duplicate_group_id": calibration_frame[
                    "duplicate_group_id"
                ].to_numpy(
                    dtype=str
                ),
                "true_score": calibration_true_scores,
            }
        )

        calibration_group_scores = (
            calibration_score_frame.groupby(
                "duplicate_group_id",
                as_index=False,
                sort=True,
            )[
                "true_score"
            ]
            .max()
        )

        for nominal_coverage in CONFORMAL_LEVELS:
            qhat, order_index = finite_sample_quantile(
                calibration_group_scores[
                    "true_score"
                ],
                nominal_coverage,
            )

            conformal_quantile_rows.append(
                {
                    "dataset_id": dataset_id,
                    "representation": representation,
                    "method": method,
                    "nominal_coverage": nominal_coverage,
                    "qhat": qhat,
                    "n_calibration_groups": len(
                        calibration_group_scores
                    ),
                    "finite_sample_order_index": order_index,
                }
            )

            prediction_sets = (
                test_scores
                <= qhat
            )

            row_covered = prediction_sets[
                np.arange(
                    len(
                        test_frame
                    )
                ),
                test_true_indices,
            ]

            row_set_size = prediction_sets.sum(
                axis=1
            )

            row_result = pd.DataFrame(
                {
                    "duplicate_group_id": test_frame[
                        "duplicate_group_id"
                    ].to_numpy(
                        dtype=str
                    ),
                    "class_label": test_frame[
                        "class_label"
                    ].to_numpy(
                        dtype=str
                    ),
                    "covered": row_covered,
                    "set_size": row_set_size,
                }
            )

            for evaluation_unit in [
                "row",
                "duplicate_group_id",
            ]:
                if evaluation_unit == "row":
                    evaluation = row_result.copy()
                else:
                    evaluation = (
                        row_result.groupby(
                            "duplicate_group_id",
                            as_index=False,
                            sort=True,
                        )
                        .agg(
                            class_label=(
                                "class_label",
                                "first",
                            ),
                            covered=(
                                "covered",
                                "min",
                            ),
                            set_size=(
                                "set_size",
                                "max",
                            ),
                        )
                    )

                coverage = float(
                    evaluation[
                        "covered"
                    ].mean()
                )

                empty_set_rate = float(
                    np.mean(
                        evaluation[
                            "set_size"
                        ].to_numpy(
                            dtype=int
                        )
                        == 0
                    )
                )

                conformal_rows.append(
                    {
                        "dataset_id": dataset_id,
                        "representation": representation,
                        "method": method,
                        "nominal_coverage": nominal_coverage,
                        "evaluation_unit": evaluation_unit,
                        "coverage": coverage,
                        "coverage_gap": (
                            coverage
                            - nominal_coverage
                        ),
                        "mean_set_size": float(
                            evaluation[
                                "set_size"
                            ].mean()
                        ),
                        "singleton_rate": float(
                            np.mean(
                                evaluation[
                                    "set_size"
                                ].to_numpy(
                                    dtype=int
                                )
                                == 1
                            )
                        ),
                        "empty_set_rate": empty_set_rate,
                        "nonempty_coverage_upper_bound": (
                            1.0
                            - empty_set_rate
                        ),
                        "margin_below_nonempty_upper_bound": (
                            (
                                1.0
                                - empty_set_rate
                            )
                            - coverage
                        ),
                        "n_evaluation_units": len(
                            evaluation
                        ),
                    }
                )

                for class_label in class_order:
                    subset = evaluation.loc[
                        evaluation[
                            "class_label"
                        ] == class_label
                    ]

                    conformal_class_rows.append(
                        {
                            "dataset_id": dataset_id,
                            "representation": representation,
                            "method": method,
                            "nominal_coverage": nominal_coverage,
                            "evaluation_unit": evaluation_unit,
                            "class_label": class_label,
                            "support": len(
                                subset
                            ),
                            "coverage": float(
                                subset[
                                    "covered"
                                ].mean()
                            ),
                            "mean_set_size": float(
                                subset[
                                    "set_size"
                                ].mean()
                            ),
                            "empty_set_rate": float(
                                np.mean(
                                    subset[
                                        "set_size"
                                    ].to_numpy(
                                        dtype=int
                                    )
                                    == 0
                                )
                            ),
                        }
                    )

conformal_results = pd.DataFrame(
    conformal_rows
)

conformal_classwise = pd.DataFrame(
    conformal_class_rows
)

conformal_quantiles = pd.DataFrame(
    conformal_quantile_rows
)

primary_aps = conformal_results.loc[
    (
        conformal_results[
            "method"
        ] == "APS_randomized"
    )
    & (
        conformal_results[
            "evaluation_unit"
        ] == "duplicate_group_id"
    )
].copy()

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(
        12,
        5,
    ),
)

for axis, metric, ylabel in [
    (
        axes[
            0
        ],
        "coverage",
        "Empirical coverage",
    ),
    (
        axes[
            1
        ],
        "mean_set_size",
        "Mean prediction-set size",
    ),
]:
    x_positions = np.arange(
        len(
            primary_aps
        )
    )

    axis.bar(
        x_positions,
        primary_aps[
            metric
        ],
    )

    axis.set_xticks(
        x_positions
    )

    axis.set_xticklabels(
        [
            (
                f"{row.dataset_id}\n"
                f"{int(row.nominal_coverage * 100)}%"
            )
            for row in primary_aps.itertuples()
        ],
        rotation=30,
        ha="right",
    )

    axis.set_ylabel(
        ylabel
    )

    axis.grid(
        True,
        axis="y",
        alpha=0.3,
    )

axes[
    0
].set_title(
    "Randomized APS coverage"
)

axes[
    1
].set_title(
    "Randomized APS efficiency"
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_5_conformal_APS_summary.{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

display(primary_aps)

In [ ]:
# ============================================================
# 10. Confusion matrices and classwise metrics
# ============================================================
classwise_rows = []
confusion_tables = {}

for dataset_id, representation in FINAL_CHAMPIONS.items():
    class_order = class_order_for(
        dataset_id,
        representation,
    )

    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ]

    probabilities = probability_matrix(
        test_frame,
        "native",
    )

    y_true = test_frame[
        "class_label"
    ].to_numpy(
        dtype=str
    )

    y_pred = probability_labels(
        probabilities,
        class_order,
    )

    matrix = confusion_matrix(
        y_true,
        y_pred,
        labels=class_order,
    )

    matrix_frame = pd.DataFrame(
        matrix,
        index=class_order,
        columns=class_order,
    )

    confusion_tables[
        dataset_id
    ] = matrix_frame

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=class_order,
            zero_division=0,
        )
    )

    for class_index, class_label in enumerate(
        class_order
    ):
        classwise_rows.append(
            {
                "dataset_id": dataset_id,
                "representation": representation,
                "class_label": class_label,
                "precision": float(
                    precision[
                        class_index
                    ]
                ),
                "recall": float(
                    recall[
                        class_index
                    ]
                ),
                "f1": float(
                    f1[
                        class_index
                    ]
                ),
                "support": int(
                    support[
                        class_index
                    ]
                ),
            }
        )

    figure, axis = plt.subplots(
        figsize=(
            8,
            7,
        )
    )

    image = axis.imshow(
        matrix,
        aspect="auto",
    )

    axis.set_xticks(
        np.arange(
            len(
                class_order
            )
        )
    )

    axis.set_yticks(
        np.arange(
            len(
                class_order
            )
        )
    )

    axis.set_xticklabels(
        class_order,
        rotation=45,
        ha="right",
    )

    axis.set_yticklabels(
        class_order,
    )

    axis.set_xlabel(
        "Predicted class"
    )

    axis.set_ylabel(
        "True class"
    )

    axis.set_title(
        (
            f"Frozen-test confusion matrix — "
            f"{dataset_id.replace('_', ' ').title()}"
        )
    )

    for row_index in range(
        matrix.shape[
            0
        ]
    ):
        for column_index in range(
            matrix.shape[
                1
            ]
        ):
            axis.text(
                column_index,
                row_index,
                str(
                    matrix[
                        row_index,
                        column_index,
                    ]
                ),
                ha="center",
                va="center",
                fontsize=8,
            )

    figure.colorbar(
        image,
        ax=axis,
        label="Count",
    )

    figure.tight_layout()

    for extension in [
        "png",
        "pdf",
    ]:
        figure.savefig(
            FIGURES_SUPP_DIR
            / (
                f"Figure_S_confusion_matrix_"
                f"{dataset_id}.{extension}"
            ),
            dpi=300
            if extension
            == "png"
            else None,
            bbox_inches="tight",
        )

    plt.show()

classwise_metrics = pd.DataFrame(
    classwise_rows
)

figure, axis = plt.subplots(
    figsize=(
        10,
        6,
    )
)

plot_frame = classwise_metrics.copy()

plot_frame[
    "display_label"
] = (
    plot_frame[
        "dataset_id"
    ]
    + ": "
    + plot_frame[
        "class_label"
    ]
)

axis.barh(
    plot_frame[
        "display_label"
    ],
    plot_frame[
        "f1"
    ],
)

axis.set_xlabel(
    "Classwise F1 score"
)

axis.set_ylabel(
    "Class"
)

axis.set_title(
    "Classwise frozen-test performance"
)

axis.grid(
    True,
    axis="x",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_SUPP_DIR
        / (
            f"Figure_S_classwise_F1.{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

display(classwise_metrics)

In [ ]:
# ============================================================
# 11. Corrected representation and controlled size–shape figures
# ============================================================
representation_metrics = source_tables[
    "representation_metrics"
].copy()

paired_contrasts = source_tables[
    "paired_contrasts"
].copy()

mcnemar_results = source_tables[
    "mcnemar"
].copy()

rice_equivalence = source_tables[
    "rice_equivalence"
].copy()

controlled_importance = source_tables[
    "controlled_importance"
].copy()

controlled_scale = source_tables[
    "controlled_scale"
].copy()

figure, axis = plt.subplots(
    figsize=(
        11,
        6,
    )
)

representations_to_plot = [
    "R1_full_native",
    "R2_measured_core",
    "R3_complete_invariant",
    "R4_complete_log_area",
]

plot_frame = representation_metrics.loc[
    representation_metrics[
        "representation"
    ].isin(
        representations_to_plot
    )
].copy()

plot_frame[
    "display"
] = (
    plot_frame[
        "dataset_id"
    ]
    + "\n"
    + plot_frame[
        "representation"
    ]
)

x_positions = np.arange(
    len(
        plot_frame
    )
)

axis.bar(
    x_positions,
    plot_frame[
        "macro_f1"
    ],
)

axis.set_xticks(
    x_positions
)

axis.set_xticklabels(
    plot_frame[
        "display"
    ],
    rotation=45,
    ha="right",
)

axis.set_ylabel(
    "Frozen-test macro-F1"
)

axis.set_title(
    "Corrected representation comparison"
)

axis.set_ylim(
    0.70,
    1.00,
)

axis.grid(
    True,
    axis="y",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_1_corrected_representation_performance."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

macro_contrasts = paired_contrasts.loc[
    paired_contrasts[
        "metric"
    ] == "macro_f1"
].copy()

key_contrasts = macro_contrasts.loc[
    (
        macro_contrasts[
            "representation_a"
        ]
        == "R1_full_native"
    )
    & (
        macro_contrasts[
            "representation_b"
        ].isin(
            [
                "R2_measured_core",
                "R3_complete_invariant",
                "R4_complete_log_area",
            ]
        )
    )
].copy()

key_contrasts[
    "label"
] = (
    key_contrasts[
        "dataset_id"
    ]
    + ": "
    + key_contrasts[
        "representation_a"
    ]
    + " − "
    + key_contrasts[
        "representation_b"
    ]
)

figure, axis = plt.subplots(
    figsize=(
        10,
        6,
    )
)

y_positions = np.arange(
    len(
        key_contrasts
    )
)

axis.errorbar(
    key_contrasts[
        "delta_a_minus_b"
    ],
    y_positions,
    xerr=np.vstack(
        [
            key_contrasts[
                "delta_a_minus_b"
            ]
            - key_contrasts[
                "ci_low"
            ],
            key_contrasts[
                "ci_high"
            ]
            - key_contrasts[
                "delta_a_minus_b"
            ],
        ]
    ),
    fmt="o",
    capsize=4,
)

axis.axvline(
    0.0,
    linestyle="--",
    linewidth=1,
)

axis.set_yticks(
    y_positions
)

axis.set_yticklabels(
    key_contrasts[
        "label"
    ]
)

axis.set_xlabel(
    "Paired macro-F1 difference"
)

axis.set_title(
    "Corrected paired representation contrasts"
)

axis.grid(
    True,
    axis="x",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_2_corrected_paired_contrasts."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

figure, axes = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=(
        12,
        5,
    ),
)

for axis, analysis, title in [
    (
        axes[
            0
        ],
        "full_block",
        "Full native blocks",
    ),
    (
        axes[
            1
        ],
        "matched_two_feature",
        "Matched two-feature blocks",
    ),
]:
    subset = controlled_importance.loc[
        controlled_importance[
            "analysis"
        ] == analysis
    ].copy()

    labels = [
        (
            f"{row.dataset_id}\n"
            f"{row.block}"
        )
        for row in subset.itertuples()
    ]

    positions = np.arange(
        len(
            subset
        )
    )

    axis.bar(
        positions,
        subset[
            "importance_mean"
        ],
        yerr=subset[
            "importance_sd"
        ],
        capsize=4,
    )

    axis.set_xticks(
        positions
    )

    axis.set_xticklabels(
        labels,
        rotation=35,
        ha="right",
    )

    axis.set_ylabel(
        "Macro-F1 decrease"
    )

    axis.set_title(
        title
    )

    axis.grid(
        True,
        axis="y",
        alpha=0.3,
    )

figure.suptitle(
    "Controlled R1 size–shape permutation analysis"
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_3_controlled_R1_block_importance."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

figure, axis = plt.subplots(
    figsize=(
        8,
        6,
    )
)

for dataset_id in sorted(
    controlled_scale[
        "dataset_id"
    ].unique()
):
    subset = controlled_scale.loc[
        controlled_scale[
            "dataset_id"
        ] == dataset_id
    ].sort_values(
        "scale_bias_percent"
    )

    axis.plot(
        subset[
            "scale_bias_percent"
        ],
        subset[
            "macro_f1_drop"
        ],
        marker="o",
        label=dataset_id.replace(
            "_",
            " ",
        ).title(),
    )

axis.axhline(
    0.02,
    linestyle="--",
    linewidth=1,
    label="Prespecified practical-loss threshold",
)

axis.set_xlabel(
    "Coherent image-space scale bias (%)"
)

axis.set_ylabel(
    "Macro-F1 decrease from zero bias"
)

axis.set_title(
    "Controlled R1 sensitivity to coherent image-space scale bias"
)

axis.grid(
    True,
    alpha=0.3,
)

axis.legend()
figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_MAIN_DIR
        / (
            f"Figure_6_controlled_R1_scale_sensitivity."
            f"{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

In [ ]:
# ============================================================
# 12. Leakage audit summaries
# ============================================================
naive_leakage = source_tables[
    "naive_leakage"
].copy()

fixed_leakage = source_tables[
    "fixed_leakage"
].copy()

naive_summary = (
    naive_leakage.groupby(
        "dataset_id",
        as_index=False,
    )
    .agg(
        n_repeats=(
            "repeat",
            "count",
        ),
        mean_naive_macro_f1=(
            "naive_macro_f1",
            "mean",
        ),
        sd_naive_macro_f1=(
            "naive_macro_f1",
            "std",
        ),
        official_grouped_macro_f1=(
            "official_grouped_macro_f1",
            "first",
        ),
        mean_naive_minus_official_macro_f1=(
            "naive_minus_official_macro_f1",
            "mean",
        ),
        minimum_naive_minus_official_macro_f1=(
            "naive_minus_official_macro_f1",
            "min",
        ),
        maximum_naive_minus_official_macro_f1=(
            "naive_minus_official_macro_f1",
            "max",
        ),
        mean_development_test_group_overlap=(
            "development_test_group_overlap_count",
            "mean",
        ),
    )
)

figure, axis = plt.subplots(
    figsize=(
        8,
        5,
    )
)

axis.bar(
    naive_summary[
        "dataset_id"
    ],
    naive_summary[
        "mean_naive_minus_official_macro_f1"
    ],
    yerr=naive_summary[
        "sd_naive_macro_f1"
    ],
    capsize=4,
)

axis.axhline(
    0.0,
    linestyle="--",
    linewidth=1,
)

axis.set_xlabel(
    "Dataset"
)

axis.set_ylabel(
    "Naive row-split minus official macro-F1"
)

axis.set_title(
    "Descriptive row-level split audit"
)

axis.grid(
    True,
    axis="y",
    alpha=0.3,
)

figure.tight_layout()

for extension in [
    "png",
    "pdf",
]:
    figure.savefig(
        FIGURES_SUPP_DIR
        / (
            f"Figure_S_naive_split_audit.{extension}"
        ),
        dpi=300
        if extension
        == "png"
        else None,
        bbox_inches="tight",
    )

plt.show()

display(naive_summary)
display(fixed_leakage)

In [ ]:
# ============================================================
# 13. Manuscript-ready table construction and export
# ============================================================
def save_table(
    frame,
    directory,
    stem,
    float_format="%.6f",
):
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    csv_path = (
        directory
        / f"{stem}.csv"
    )

    md_path = (
        directory
        / f"{stem}.md"
    )

    tex_path = (
        directory
        / f"{stem}.tex"
    )

    frame.to_csv(
        csv_path,
        index=False,
        encoding="utf-8-sig",
    )

    md_path.write_text(
        frame.to_markdown(
            index=False,
            floatfmt=".6f",
        ),
        encoding="utf-8",
    )

    tex_path.write_text(
        frame.to_latex(
            index=False,
            float_format=lambda value: (
                float_format
                % value
            ),
            escape=True,
        ),
        encoding="utf-8",
    )

    return {
        "csv": csv_path,
        "markdown": md_path,
        "latex": tex_path,
    }

# Main Table 1: datasets and frozen models.
representation_audit = source_tables[
    "representation_audit"
]

dataset_rows = []

for dataset_id, representation in FINAL_CHAMPIONS.items():
    audit_row = representation_audit.loc[
        (
            representation_audit[
                "dataset_id"
            ] == dataset_id
        )
        & (
            representation_audit[
                "representation"
            ] == representation
        )
    ].iloc[0]

    test_frame = prediction_frames[
        (
            dataset_id,
            "test",
        )
    ]

    calibration_frame = prediction_frames[
        (
            dataset_id,
            "calibration",
        )
    ]

    total_rows = int(
        audit_row[
            "rows"
        ]
    )

    test_rows = len(
        test_frame
    )

    calibration_rows_count = len(
        calibration_frame
    )

    train_rows = (
        total_rows
        - test_rows
        - calibration_rows_count
    )

    dataset_rows.append(
        {
            "Dataset": dataset_id,
            "Total rows": total_rows,
            "Training rows": train_rows,
            "Calibration rows": calibration_rows_count,
            "Test rows": test_rows,
            "Classes": test_frame[
                "class_label"
            ].nunique(),
            "Frozen primary representation": representation,
            "Predictors": int(
                audit_row[
                    "n_features"
                ]
            ),
            "Measurement domain": (
                "pixels, squared pixels, and dimensionless ratios"
            ),
        }
    )

main_table_1 = pd.DataFrame(
    dataset_rows
)

# Main Table 2: champion performance.
row_point_metrics = champion_metrics.loc[
    champion_metrics[
        "evaluation_unit"
    ] == "row"
].copy()

main_table_2 = (
    row_point_metrics[
        [
            "dataset_id",
            "representation",
            "n_units",
            "accuracy",
            "balanced_accuracy",
            "macro_f1",
            "mcc",
            "log_loss",
            "brier_multiclass",
        ]
    ]
    .rename(
        columns={
            "dataset_id": "Dataset",
            "representation": "Representation",
            "n_units": "Test rows",
            "accuracy": "Accuracy",
            "balanced_accuracy": "Balanced accuracy",
            "macro_f1": "Macro-F1",
            "mcc": "MCC",
            "log_loss": "Log-loss",
            "brier_multiclass": "Multiclass Brier",
        }
    )
)

# Main Table 3: corrected representation contrasts.
main_table_3 = key_contrasts[
    [
        "dataset_id",
        "representation_a",
        "representation_b",
        "delta_a_minus_b",
        "ci_low",
        "ci_high",
        "bootstrap_p_raw",
        "bootstrap_p_holm",
        "reject_holm_0_05",
    ]
].rename(
    columns={
        "dataset_id": "Dataset",
        "representation_a": "Representation A",
        "representation_b": "Representation B",
        "delta_a_minus_b": "Delta macro-F1 (A-B)",
        "ci_low": "95% CI lower",
        "ci_high": "95% CI upper",
        "bootstrap_p_raw": "Bootstrap p",
        "bootstrap_p_holm": "Holm-adjusted p",
        "reject_holm_0_05": "Reject at 0.05",
    }
)

# Main Table 4: selective classification.
main_table_4 = selective_results.loc[
    selective_results[
        "evaluation_unit"
    ] == "duplicate_group_id"
][
    [
        "dataset_id",
        "representation",
        "certainty_score",
        "target_calibration_coverage",
        "certainty_threshold",
        "coverage",
        "selective_accuracy",
        "selective_macro_f1",
        "selective_risk",
        "risk_reduction",
    ]
].rename(
    columns={
        "dataset_id": "Dataset",
        "representation": "Representation",
        "certainty_score": "Certainty score",
        "target_calibration_coverage": "Calibration target",
        "certainty_threshold": "Threshold",
        "coverage": "Test coverage",
        "selective_accuracy": "Selective accuracy",
        "selective_macro_f1": "Selective macro-F1",
        "selective_risk": "Selective risk",
        "risk_reduction": "Risk reduction",
    }
)

# Main Table 5: randomized APS.
main_table_5 = primary_aps[
    [
        "dataset_id",
        "representation",
        "nominal_coverage",
        "coverage",
        "coverage_gap",
        "mean_set_size",
        "singleton_rate",
        "empty_set_rate",
        "nonempty_coverage_upper_bound",
        "margin_below_nonempty_upper_bound",
    ]
].rename(
    columns={
        "dataset_id": "Dataset",
        "representation": "Representation",
        "nominal_coverage": "Nominal coverage",
        "coverage": "Observed coverage",
        "coverage_gap": "Coverage gap",
        "mean_set_size": "Mean set size",
        "singleton_rate": "Singleton rate",
        "empty_set_rate": "Empty-set rate",
        "nonempty_coverage_upper_bound": "1 - empty-set rate",
        "margin_below_nonempty_upper_bound": (
            "Margin below nonempty upper bound"
        ),
    }
)

# Main Table 6: controlled R1 block importance.
main_table_6 = controlled_importance[
    [
        "dataset_id",
        "analysis",
        "block",
        "n_block_features",
        "matched_subset_size",
        "importance_mean",
        "importance_sd",
        "importance_per_feature",
    ]
].rename(
    columns={
        "dataset_id": "Dataset",
        "analysis": "Analysis",
        "block": "Feature block",
        "n_block_features": "Native block size",
        "matched_subset_size": "Matched subset size",
        "importance_mean": "Mean macro-F1 decrease",
        "importance_sd": "Permutation SD",
        "importance_per_feature": "Decrease per permuted feature",
    }
)

# Main Table 7: selected coherent scale conditions.
main_table_7 = controlled_scale.loc[
    controlled_scale[
        "scale_bias_percent"
    ].isin(
        [
            -5.0,
            -2.0,
            0.0,
            2.0,
            5.0,
        ]
    )
][
    [
        "dataset_id",
        "representation",
        "scale_bias_percent",
        "macro_f1",
        "macro_f1_drop",
        "accuracy",
        "mcc",
        "label_flip_rate",
    ]
].rename(
    columns={
        "dataset_id": "Dataset",
        "representation": "Representation",
        "scale_bias_percent": "Image-space scale bias (%)",
        "macro_f1": "Macro-F1",
        "macro_f1_drop": "Macro-F1 decrease",
        "accuracy": "Accuracy",
        "mcc": "MCC",
        "label_flip_rate": "Label-flip rate",
    }
)

main_tables = {
    "Table_1_dataset_and_frozen_models": main_table_1,
    "Table_2_frozen_test_performance": main_table_2,
    "Table_3_corrected_representation_contrasts": main_table_3,
    "Table_4_selective_classification": main_table_4,
    "Table_5_randomized_APS": main_table_5,
    "Table_6_controlled_R1_block_importance": main_table_6,
    "Table_7_controlled_R1_scale_sensitivity": main_table_7,
}

supplementary_tables = {
    "Table_S1_complete_representation_audit": (
        representation_audit
    ),
    "Table_S2_all_training_candidates": (
        source_tables[
            "all_candidates"
        ]
    ),
    "Table_S3_selected_candidate_per_representation": (
        source_tables[
            "selected_candidates"
        ]
    ),
    "Table_S4_all_test_representation_metrics": (
        representation_metrics
    ),
    "Table_S5_all_paired_representation_contrasts": (
        paired_contrasts
    ),
    "Table_S6_exact_McNemar_results": (
        mcnemar_results
    ),
    "Table_S7_rice_equivalence_results": (
        rice_equivalence
    ),
    "Table_S8_champion_metric_intervals": (
        champion_metric_intervals
    ),
    "Table_S9_calibration_audit": (
        calibration_audit
    ),
    "Table_S10_selective_row_and_group_results": (
        selective_results
    ),
    "Table_S11_risk_coverage_curve_values": (
        risk_coverage_results
    ),
    "Table_S12_all_conformal_results": (
        conformal_results
    ),
    "Table_S13_conformal_classwise_results": (
        conformal_classwise
    ),
    "Table_S14_conformal_quantiles": (
        conformal_quantiles
    ),
    "Table_S15_complete_classwise_metrics": (
        classwise_metrics
    ),
    "Table_S16_controlled_R1_importance": (
        controlled_importance
    ),
    "Table_S17_complete_R1_scale_sensitivity": (
        controlled_scale
    ),
    "Table_S18_naive_split_audit": (
        naive_leakage
    ),
    "Table_S19_naive_split_summary": (
        naive_summary
    ),
    "Table_S20_fixed_test_duplicate_contamination": (
        fixed_leakage
    ),
    "Table_S21_computational_timing": (
        source_tables[
            "timing"
        ]
    ),
}

for table_name, table_frame in main_tables.items():
    save_table(
        table_frame,
        TABLES_MAIN_DIR,
        table_name,
    )

for table_name, table_frame in supplementary_tables.items():
    save_table(
        table_frame,
        TABLES_SUPP_DIR,
        table_name,
    )

for dataset_id, matrix_frame in confusion_tables.items():
    matrix_with_label = (
        matrix_frame.reset_index()
        .rename(
            columns={
                "index": "True class"
            }
        )
    )

    save_table(
        matrix_with_label,
        TABLES_SUPP_DIR,
        (
            f"Table_S_confusion_matrix_{dataset_id}"
        ),
        float_format="%.0f",
    )

print(
    "Main tables exported:",
    len(
        main_tables
    ),
)

print(
    "Supplementary tables exported:",
    len(
        supplementary_tables
    )
    + len(
        confusion_tables
    ),
)

In [ ]:
# ============================================================
# 14. Numerical claim register and reviewer-response map
# ============================================================
def point_metric(
    dataset_id,
    metric,
):
    return float(
        champion_metrics.loc[
            (
                champion_metrics[
                    "dataset_id"
                ] == dataset_id
            )
            & (
                champion_metrics[
                    "evaluation_unit"
                ] == "row"
            ),
            metric,
        ].iloc[0]
    )

def contrast_row(
    dataset_id,
    representation_a,
    representation_b,
):
    return paired_contrasts.loc[
        (
            paired_contrasts[
                "dataset_id"
            ] == dataset_id
        )
        & (
            paired_contrasts[
                "representation_a"
            ] == representation_a
        )
        & (
            paired_contrasts[
                "representation_b"
            ] == representation_b
        )
        & (
            paired_contrasts[
                "metric"
            ] == "macro_f1"
        )
    ].iloc[0]

dry_r1_r3 = contrast_row(
    "dry_bean",
    "R1_full_native",
    "R3_complete_invariant",
)

dry_r1_r4 = contrast_row(
    "dry_bean",
    "R1_full_native",
    "R4_complete_log_area",
)

rice_r1_r3 = contrast_row(
    "iniap_rice",
    "R1_full_native",
    "R3_complete_invariant",
)

claim_rows = [
    {
        "claim_id": "C001",
        "section": "Results—primary performance",
        "claim": (
            "The frozen Dry Bean R1 model achieved "
            f"macro-F1 = {point_metric('dry_bean', 'macro_f1'):.6f}."
        ),
        "source": (
            "Table_2_frozen_test_performance"
        ),
    },
    {
        "claim_id": "C002",
        "section": "Results—primary performance",
        "claim": (
            "The frozen INIAP Rice R4 model achieved "
            f"macro-F1 = {point_metric('iniap_rice', 'macro_f1'):.6f}."
        ),
        "source": (
            "Table_2_frozen_test_performance"
        ),
    },
    {
        "claim_id": "C003",
        "section": "Results—representations",
        "claim": (
            "In Dry Bean, R1 exceeded the complete invariant "
            f"representation by {dry_r1_r3['delta_a_minus_b']:.6f} "
            "macro-F1."
        ),
        "source": (
            "Table_3_corrected_representation_contrasts"
        ),
    },
    {
        "claim_id": "C004",
        "section": "Results—representations",
        "claim": (
            "The 95% group-bootstrap interval for the corrected "
            "Dry Bean R1–R3-complete contrast was "
            f"{dry_r1_r3['ci_low']:.6f} to "
            f"{dry_r1_r3['ci_high']:.6f}."
        ),
        "source": (
            "Table_3_corrected_representation_contrasts"
        ),
    },
    {
        "claim_id": "C005",
        "section": "Results—representations",
        "claim": (
            "Adding log-pixel-area to the complete invariant "
            "Dry Bean representation reduced the R1 gap to "
            f"{dry_r1_r4['delta_a_minus_b']:.6f} macro-F1."
        ),
        "source": (
            "Table_3_corrected_representation_contrasts"
        ),
    },
    {
        "claim_id": "C006",
        "section": "Results—rice equivalence",
        "claim": (
            "The native and invariant rice representations were "
            "assessed using a prespecified ±0.01 macro-F1 "
            "equivalence margin."
        ),
        "source": (
            "Table_S7_rice_equivalence_results"
        ),
    },
    {
        "claim_id": "C007",
        "section": "Discussion—interpretation",
        "claim": (
            "Scale-dependent variables can be strongly used by a "
            "native-feature rice model even when an invariant model "
            "retains practically equivalent predictive performance."
        ),
        "source": (
            "Table_6_controlled_R1_block_importance; "
            "Table_S7_rice_equivalence_results"
        ),
    },
    {
        "claim_id": "C008",
        "section": "Methods—measurement domain",
        "claim": (
            "Linear and area-related variables were analyzed in "
            "pixels and squared pixels, respectively; no physical-unit "
            "conversion was assumed."
        ),
        "source": (
            "final_analysis_freeze_protocol.json"
        ),
    },
    {
        "claim_id": "C009",
        "section": "Methods—conformal prediction",
        "claim": (
            "Empty conformal prediction sets were retained and counted "
            "as non-covering outcomes."
        ),
        "source": (
            "final_analysis_freeze_protocol.json; "
            "Table_5_randomized_APS"
        ),
    },
    {
        "claim_id": "C010",
        "section": "Discussion—calibration",
        "claim": (
            "Temperature scaling was treated as a diagnostic layer, "
            "not as a guaranteed performance improvement."
        ),
        "source": (
            "Table_S9_calibration_audit"
        ),
    },
]

numerical_claim_register = pd.DataFrame(
    claim_rows
)

numerical_claim_register.to_csv(
    CLAIMS_DIR
    / "numerical_claim_register.csv",
    index=False,
    encoding="utf-8-sig",
)

reviewer_response_map = pd.DataFrame(
    [
        {
            "Reviewer item": "M1",
            "Resolution": "Resolved computationally",
            "Evidence": (
                "Tables 1, 3, S1, S4–S6; Figures 1–2"
            ),
            "Required manuscript action": (
                "Define complete and reduced invariant representations "
                "separately and rewrite the central contrast."
            ),
        },
        {
            "Reviewer item": "M2",
            "Resolution": "Resolved computationally",
            "Evidence": (
                "Tables 6–7, S16–S17; Figures 3 and 6"
            ),
            "Required manuscript action": (
                "Distinguish model reliance from representation sufficiency."
            ),
        },
        {
            "Reviewer item": "M3",
            "Resolution": "Resolved computationally",
            "Evidence": "Tables 3, S5, and S6",
            "Required manuscript action": (
                "Report bootstrap metric inference and exact McNemar "
                "inference separately."
            ),
        },
        {
            "Reviewer item": "M4",
            "Resolution": "Editorial action pending",
            "Evidence": "",
            "Required manuscript action": (
                "Add a verified benchmark comparison from real published "
                "sources; do not populate from memory."
            ),
        },
        {
            "Reviewer item": "M5",
            "Resolution": "Clarified analytically",
            "Evidence": (
                "Final analysis-freeze protocol and Table 1"
            ),
            "Required manuscript action": (
                "State pixel-based measurement units and provide the real "
                "repository URL and license."
            ),
        },
        {
            "Reviewer item": "M6",
            "Resolution": "Resolved computationally",
            "Evidence": "Tables 3, S3–S5; Figures 1–2",
            "Required manuscript action": "Report and discuss R2.",
        },
        {
            "Reviewer item": "M7",
            "Resolution": "Resolved computationally",
            "Evidence": "Table S7",
            "Required manuscript action": (
                "Use equivalence language only when the prespecified test "
                "supports it."
            ),
        },
        {
            "Reviewer item": "M8",
            "Resolution": "Resolved computationally",
            "Evidence": (
                "Table S9 and supplementary reliability figures"
            ),
            "Required manuscript action": (
                "Describe temperature scaling as marginal or neutral when "
                "supported by the diagnostics."
            ),
        },
        {
            "Reviewer item": "M9",
            "Resolution": "Resolved computationally",
            "Evidence": "Tables 5, S12–S14; Figure 5",
            "Required manuscript action": (
                "Discuss empty sets, the nonempty upper bound, and the "
                "fragility of the rice operating point."
            ),
        },
        {
            "Reviewer item": "M10",
            "Resolution": "Resolved computationally",
            "Evidence": "Tables S18–S20",
            "Required manuscript action": (
                "Present group-aware splitting as a validity safeguard and "
                "quantify the limited direct duplicate effect."
            ),
        },
        {
            "Reviewer item": "M11",
            "Resolution": "Resolved computationally",
            "Evidence": "Tables 4 and S10",
            "Required manuscript action": (
                "State the evaluation unit for every selective metric."
            ),
        },
        {
            "Reviewer item": "M12",
            "Resolution": "Resolved computationally",
            "Evidence": (
                "Table S15, confusion-matrix tables, and supplementary figures"
            ),
            "Required manuscript action": (
                "Reference classwise metrics and confusion matrices."
            ),
        },
        {
            "Reviewer item": "M13",
            "Resolution": "Editorial action pending",
            "Evidence": "",
            "Required manuscript action": (
                "Integrate verified 2023–2026 literature into the "
                "Introduction and Discussion."
            ),
        },
        {
            "Reviewer item": "M14",
            "Resolution": "Repository action pending",
            "Evidence": (
                "Output ZIP, claim register, and SHA-256 manifest"
            ),
            "Required manuscript action": (
                "Deposit the actual code and permitted data, obtain a DOI, "
                "and state the license."
            ),
        },
    ]
)

reviewer_response_map.to_csv(
    RESULTS_DIR
    / "reviewer_response_evidence_map.csv",
    index=False,
    encoding="utf-8-sig",
)

manual_fields_required = pd.DataFrame(
    [
        {
            "Field": "Authors",
            "Status": "Required before submission",
        },
        {
            "Field": "Affiliations",
            "Status": "Required before submission",
        },
        {
            "Field": "Corresponding author",
            "Status": "Required before submission",
        },
        {
            "Field": "CRediT contributions",
            "Status": "Required before submission",
        },
        {
            "Field": "Funding",
            "Status": "Required before submission",
        },
        {
            "Field": "GitHub or repository URL",
            "Status": "Required before submission",
        },
        {
            "Field": "Repository DOI",
            "Status": "Required before submission",
        },
        {
            "Field": "Data and code license",
            "Status": "Required before submission",
        },
        {
            "Field": "Conflicts of interest",
            "Status": "Required before submission",
        },
        {
            "Field": "Verified benchmark-comparison references",
            "Status": "Required before submission",
        },
        {
            "Field": "Verified recent 2023–2026 references",
            "Status": "Required before submission",
        },
    ]
)

manual_fields_required.to_csv(
    RESULTS_DIR
    / "manual_fields_required_before_submission.csv",
    index=False,
    encoding="utf-8-sig",
)

display(numerical_claim_register)
display(reviewer_response_map)

In [ ]:
# ============================================================
# 15. Final freeze protocol, validation, and integrity manifest
# ============================================================
freeze_protocol = {
    "protocol_name": (
        "Grain_AI_MDPI_final_analysis_freeze_v1"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "source_zip": SOURCE_ZIP_NAME,
    "source_zip_sha256": sha256_file(
        SOURCE_ZIP_PATH
    ),
    "source_notebook12_protocol": source_protocol,
    "frozen_primary_models": FINAL_CHAMPIONS,
    "primary_model_policy": {
        "dry_bean": (
            "Retain the prespecified original R1 primary model. "
            "Notebook 12 representation results are confirmatory "
            "and do not trigger post-test reselection."
        ),
        "iniap_rice": (
            "Retain the complete invariant plus log-pixel-area "
            "primary model."
        ),
    },
    "probability_label_rule": (
        "Argmax of probability columns aligned to the metadata "
        "class order; model.predict is not used."
    ),
    "bootstrap": {
        "unit": "duplicate_group_id",
        "stratification": "class_label",
        "repetitions": N_BOOTSTRAP,
        "confidence_level": CONFIDENCE_LEVEL,
        "seed": FINAL_SEED,
    },
    "selective_classification": {
        "calibration_unit": "duplicate_group_id",
        "target_coverage": SELECTIVE_TARGET_COVERAGE,
        "scores": CERTAINTY_SCORES,
        "threshold_source": "calibration partition only",
    },
    "conformal_prediction": {
        "methods": CONFORMAL_METHODS,
        "nominal_levels": CONFORMAL_LEVELS,
        "calibration_unit": "duplicate_group_id",
        "randomization_seed": CONFORMAL_SEED,
        "empty_sets": (
            "retained and counted as non-covering"
        ),
    },
    "measurement_units": {
        "linear": "pixels",
        "area": "squared pixels",
        "shape_ratios": "dimensionless",
        "physical_unit_conversion": None,
    },
    "interpretive_constraints": [
        (
            "No absolute physical-size interpretation is made."
        ),
        (
            "No numerical image-space size is compared directly "
            "between datasets."
        ),
        (
            "Model reliance and representation sufficiency are "
            "reported as different concepts."
        ),
        (
            "Permutation SD is not described as a sampling "
            "confidence interval."
        ),
        (
            "No external benchmark value, repository DOI, or "
            "bibliographic entry is fabricated."
        ),
        (
            "No new model or representation is selected using the "
            "frozen test set."
        ),
    ],
}

freeze_protocol_path = (
    PROTOCOL_DIR
    / "final_analysis_freeze_protocol.json"
)

freeze_protocol_path.write_text(
    json.dumps(
        freeze_protocol,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

expected_main_table_stems = set(
    main_tables.keys()
)

actual_main_table_stems = {
    path.stem
    for path in TABLES_MAIN_DIR.glob(
        "*.csv"
    )
}

required_main_figures = [
    "Figure_1_corrected_representation_performance.png",
    "Figure_2_corrected_paired_contrasts.png",
    "Figure_3_controlled_R1_block_importance.png",
    "Figure_4_risk_coverage_curves.png",
    "Figure_5_conformal_APS_summary.png",
    "Figure_6_controlled_R1_scale_sensitivity.png",
]

validation_checks = {
    "notebook12_full_mode": (
        source_protocol.get(
            "run_mode"
        )
        == "FULL"
    ),
    "notebook12_validation_passed": bool(
        source_validation[
            "passed"
        ].astype(bool).all()
    ),
    "notebook12_manifest_validated": bool(
        source_integrity_validation[
            "passed"
        ].all()
    ),
    "dry_bean_primary_model_frozen_as_R1": (
        FINAL_CHAMPIONS[
            "dry_bean"
        ]
        == "R1_full_native"
    ),
    "rice_primary_model_frozen_as_R4_complete": (
        FINAL_CHAMPIONS[
            "iniap_rice"
        ]
        == "R4_complete_log_area"
    ),
    "dry_bean_complete_invariant_has_eight_features": (
        int(
            representation_audit.loc[
                (
                    representation_audit[
                        "dataset_id"
                    ] == "dry_bean"
                )
                & (
                    representation_audit[
                        "representation"
                    ] == "R3_complete_invariant"
                ),
                "n_features",
            ].iloc[0]
        )
        == 8
    ),
    "champion_prediction_frames_loaded": (
        len(
            prediction_frames
        )
        == 4
    ),
    "champion_metrics_complete": (
        len(
            champion_metrics
        )
        == 4
    ),
    "bootstrap_intervals_complete": (
        len(
            champion_metric_intervals
        )
        == len(
            FINAL_CHAMPIONS
        )
        * 6
    ),
    "calibration_audit_complete": (
        len(
            calibration_audit
        )
        == len(
            FINAL_CHAMPIONS
        )
        * 2
        * 2
    ),
    "selective_row_and_group_units_complete": (
        set(
            selective_results[
                "evaluation_unit"
            ]
        )
        == {
            "row",
            "duplicate_group_id",
        }
    ),
    "conformal_methods_complete": (
        set(
            conformal_results[
                "method"
            ]
        )
        == set(
            CONFORMAL_METHODS
        )
    ),
    "conformal_levels_complete": (
        set(
            conformal_results[
                "nominal_coverage"
            ]
        )
        == set(
            CONFORMAL_LEVELS
        )
    ),
    "empty_sets_explicit": (
        "empty_set_rate"
        in conformal_results.columns
    ),
    "classwise_metrics_complete": (
        classwise_metrics[
            "dataset_id"
        ].nunique()
        == len(
            FINAL_CHAMPIONS
        )
    ),
    "seven_main_tables_created": (
        actual_main_table_stems
        == expected_main_table_stems
    ),
    "six_main_figures_created": all(
        (
            FIGURES_MAIN_DIR
            / filename
        ).exists()
        for filename in required_main_figures
    ),
    "reviewer_map_has_M1_to_M14": (
        set(
            reviewer_response_map[
                "Reviewer item"
            ]
        )
        == {
            f"M{index}"
            for index in range(
                1,
                15,
            )
        }
    ),
    "numerical_claim_register_created": (
        len(
            numerical_claim_register
        )
        >= 10
    ),
    "measurement_units_are_image_space": (
        freeze_protocol[
            "measurement_units"
        ][
            "physical_unit_conversion"
        ]
        is None
    ),
    "no_external_values_fabricated": True,
    "no_post_test_model_reselection": True,
}

final_validation = pd.DataFrame(
    [
        {
            "check": check,
            "passed": bool(
                passed
            ),
        }
        for check, passed
        in validation_checks.items()
    ]
)

final_validation.to_csv(
    RESULTS_DIR
    / "final_notebook13_validation.csv",
    index=False,
    encoding="utf-8-sig",
)

if not final_validation[
    "passed"
].all():
    display(
        final_validation.loc[
            ~final_validation[
                "passed"
            ]
        ]
    )

    raise RuntimeError(
        "At least one Notebook 13 final validation failed."
    )

manifest_rows = []

for path in sorted(
    OUTPUT_ROOT.rglob("*")
):
    if path.is_file():
        manifest_rows.append(
            {
                "relative_path": str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(
                    path
                ),
            }
        )

integrity_manifest = pd.DataFrame(
    manifest_rows
)

integrity_manifest.to_csv(
    RESULTS_DIR
    / "notebook13_output_integrity_manifest.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Notebook 13 final validation: PASSED")
display(final_validation)
print(
    "Files registered before manifest creation:",
    len(
        integrity_manifest
    ),
)

In [ ]:
# ============================================================
# 16. Export final outputs to Google Drive
# ============================================================
DRIVE_OUTPUT_DIR = (
    DRIVE_PROJECT_ROOT
    / "NOTEBOOK13_FINAL_FREEZE_REVISED_MANUSCRIPT_ASSETS"
)

DRIVE_OUTPUT_ZIP = (
    DRIVE_PROJECT_ROOT
    / (
        "NOTEBOOK13_FINAL_FREEZE_"
        "REVISED_MANUSCRIPT_ASSETS.zip"
    )
)

if DRIVE_OUTPUT_DIR.exists():
    shutil.rmtree(
        DRIVE_OUTPUT_DIR
    )

shutil.copytree(
    OUTPUT_ROOT,
    DRIVE_OUTPUT_DIR,
)

if DRIVE_OUTPUT_ZIP.exists():
    DRIVE_OUTPUT_ZIP.unlink()

with zipfile.ZipFile(
    DRIVE_OUTPUT_ZIP,
    mode="w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as archive:
    for path in sorted(
        OUTPUT_ROOT.rglob("*")
    ):
        if path.is_file():
            archive.write(
                path,
                arcname=str(
                    path.relative_to(
                        OUTPUT_ROOT
                    )
                ),
            )

print(
    "Final output directory:",
    DRIVE_OUTPUT_DIR,
)

print(
    "Final output ZIP:",
    DRIVE_OUTPUT_ZIP,
)

print(
    "ZIP size (MB):",
    round(
        DRIVE_OUTPUT_ZIP.stat().st_size
        / 1e6,
        2,
    ),
)

print(
    "\nRequired completion message:"
)

print(
    "Notebook 13 final validation: PASSED"
)

# Post-execution decision

The computational analysis is frozen when the final cell reports:

```text
Notebook 13 final validation: PASSED
```

The next deliverables should then be prepared without additional model selection:

1. **Revised English manuscript for AI—MDPI.**
2. **English Supplementary Materials with editable equations.**
3. **Point-by-point response to the reviewer.**

The final narrative must distinguish:

- **representation sufficiency:** whether a representation can achieve comparable predictive performance;
- **model reliance:** whether a model uses a block strongly when that block is available.

The rice results must not be summarized as “the model does not use scale.” The controlled R1 analysis shows that scale-dependent variables can be strongly used, while the equivalence analysis shows that they are not uniquely necessary for high predictive performance.